In [1]:
# ============================================================
# NeuroCore Universal iEEG Loader v3
# Robust multi-format extractor
# ============================================================

import os
import numpy as np

SUPPORTED_EXT = (
    ".edf", ".bdf", ".gdf",
    ".vhdr", ".eeg",
    ".set",
    ".fif",
    ".nwb",
    ".mat"
)

def find_ieeg_files(root):
    files = []
    for path, dirs, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(SUPPORTED_EXT):
                files.append(os.path.join(path, f))
    return files


def load_file(filepath):

    ext = filepath.lower().split(".")[-1]

    # -------- EDF / BDF / GDF --------
    if ext in ["edf","bdf","gdf"]:
        try:
            import mne
            raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
            data = raw.get_data()
            sfreq = raw.info["sfreq"]
            ch_names = raw.ch_names
            return data, sfreq, ch_names, "mne_edf"
        except:
            pass

    # -------- BrainVision --------
    if ext == "vhdr":
        try:
            import mne
            raw = mne.io.read_raw_brainvision(filepath, preload=True, verbose=False)
            data = raw.get_data()
            sfreq = raw.info["sfreq"]
            ch_names = raw.ch_names
            return data, sfreq, ch_names, "brainvision"
        except:
            pass

    # -------- EEGLAB --------
    if ext == "set":
        try:
            import mne
            raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
            data = raw.get_data()
            sfreq = raw.info["sfreq"]
            ch_names = raw.ch_names
            return data, sfreq, ch_names, "eeglab"
        except:
            pass

    # -------- FIF --------
    if ext == "fif":
        try:
            import mne
            raw = mne.io.read_raw_fif(filepath, preload=True, verbose=False)
            data = raw.get_data()
            sfreq = raw.info["sfreq"]
            ch_names = raw.ch_names
            return data, sfreq, ch_names, "fif"
        except:
            pass

    # -------- NWB --------
    if ext == "nwb":
        try:
            from pynwb import NWBHDF5IO
            io = NWBHDF5IO(filepath, "r")
            nwb = io.read()

            es = list(nwb.acquisition.values())[0]
            data = np.array(es.data).T
            sfreq = es.rate
            ch_names = [f"ch_{i}" for i in range(data.shape[0])]

            return data, sfreq, ch_names, "nwb"
        except:
            pass

    # -------- MAT --------
    if ext == "mat":
        try:
            import scipy.io
            mat = scipy.io.loadmat(filepath)

            for key in mat:
                val = mat[key]
                if isinstance(val, np.ndarray) and val.ndim == 2:
                    data = val
                    sfreq = 1000
                    ch_names = [f"ch_{i}" for i in range(data.shape[0])]
                    return data, sfreq, ch_names, "mat"
        except:
            pass

    return None, None, None, "failed"


def load_ieeg_dataset(root):

    files = find_ieeg_files(root)

    dataset = []

    print(f"Files trovati: {len(files)}")

    for i, f in enumerate(files):

        data, sfreq, ch_names, loader = load_file(f)

        if data is None:
            print(f"FAILED: {f}")
            continue

        dataset.append({
            "file": f,
            "data": data.astype(np.float32),
            "sfreq": sfreq,
            "channels": ch_names,
            "loader": loader
        })

        print(f"OK [{i+1}/{len(files)}] {loader} | shape {data.shape} | sfreq {sfreq}")

    print(f"\nDataset caricati: {len(dataset)}")

    return dataset


# === USA QUI ===

DATASET_PATH = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691"

dataset = load_ieeg_dataset(DATASET_PATH)


Files trovati: 32
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-01/ieeg/._sub-01_task-countingDeviants_ieeg.eeg


/var/folders/z2/xpf0ksm94yj0mbc7tw831gcc0000gn/T/ipykernel_30112/2652909201.py:47: RuntimeWarning: MNE-Python currently only supports header versions 1.0 and 2.0, got unparsable '\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X        \x00\x02\x00\x00\x00\t\x00\x00\x002\x00\x00\x0e\x00\x00\x00\x02\x00\x00\x0e\x00\x00\x01\x1e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTR\x00\x00\x00\x01\x00\x00\x0e\x00\x00\x00\x00\x00\x00\x15\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x15\x00\x00\x15com.apple.quarantine\x000081;699ccee1;Chrome;\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-01/ieeg/._sub-01_task-countingDeviants_ieeg.vhdr
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-01/ieeg/sub-01_task-countingDeviants_ieeg.eeg
OK [4/32] brainvision | shape (7, 5812600) | sfreq 2500.0
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-02/ieeg/._sub-02_task-countingDeviants_ieeg.eeg
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-02/ieeg/._sub-02_task-countingDeviants_ieeg.vhdr
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-02/ieeg/sub-02_task-countingDeviants_ieeg.eeg


/var/folders/z2/xpf0ksm94yj0mbc7tw831gcc0000gn/T/ipykernel_30112/2652909201.py:47: RuntimeWarning: MNE-Python currently only supports header versions 1.0 and 2.0, got unparsable '\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X        \x00\x02\x00\x00\x00\t\x00\x00\x002\x00\x00\x0e\x00\x00\x00\x02\x00\x00\x0e\x00\x00\x01\x1e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTR\x00\x00\x00\x01\x00\x00\x0e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x15\x00\x00\x15com.apple.quarantine\x00\x00\x00\x00\x00\x00\x00\x00\x00%com.apple.metadata:kMDItemWhereFroms\x000081;699ccf00;Chrome;bplist00\x01\x02_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download\x08\x0bM\x00\x00\x00\x00\x00\x00\x01\x01\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

OK [8/32] brainvision | shape (7, 6897900) | sfreq 2500.0
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-03/ieeg/._sub-03_task-countingDeviants_ieeg.eeg
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-03/ieeg/._sub-03_task-countingDeviants_ieeg.vhdr
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-03/ieeg/sub-03_task-countingDeviants_ieeg.eeg


/var/folders/z2/xpf0ksm94yj0mbc7tw831gcc0000gn/T/ipykernel_30112/2652909201.py:47: RuntimeWarning: MNE-Python currently only supports header versions 1.0 and 2.0, got unparsable '\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X        \x00\x02\x00\x00\x00\t\x00\x00\x002\x00\x00\x0e\x00\x00\x00\x02\x00\x00\x0e\x00\x00\x01\x1e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTR\x00\x00\x00\x01\x00\x00\x0e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x15\x00\x00\x15com.apple.quarantine\x00\x00\x00\x00\x00\x00\x00\x00\x00%com.apple.metadata:kMDItemWhereFroms\x000081;699ccf1f;Chrome;bplist00\x01\x02_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download\x08\x0bM\x00\x00\x00\x00\x00\x00\x01\x01\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

... (7 outputs left out)

/var/folders/z2/xpf0ksm94yj0mbc7tw831gcc0000gn/T/ipykernel_30112/2652909201.py:47: RuntimeWarning: MNE-Python currently only supports header versions 1.0 and 2.0, got unparsable '\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X        \x00\x02\x00\x00\x00\t\x00\x00\x002\x00\x00\x0e\x00\x00\x00\x02\x00\x00\x0e\x00\x00\x01\x1e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTR\x00\x00\x00\x01\x00\x00\x0e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x15\x00\x00\x15com.apple.quarantine\x00\x00\x00\x00\x00\x00\x00\x00\x00%com.apple.metadata:kMDItemWhereFroms\x000081;699ccfa0;Chrome;bplist00\x01\x02_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download\x08\x0bM\x00\x00\x00\x00\x00\x00\x01\x01\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

OK [28/32] brainvision | shape (7, 9525300) | sfreq 2500.0
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-10/ieeg/._sub-10_task-countingDeviants_ieeg.eeg
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-10/ieeg/._sub-10_task-countingDeviants_ieeg.vhdr
FAILED: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-10/ieeg/sub-10_task-countingDeviants_ieeg.eeg


/var/folders/z2/xpf0ksm94yj0mbc7tw831gcc0000gn/T/ipykernel_30112/2652909201.py:47: RuntimeWarning: MNE-Python currently only supports header versions 1.0 and 2.0, got unparsable '\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X        \x00\x02\x00\x00\x00\t\x00\x00\x002\x00\x00\x0e\x00\x00\x00\x02\x00\x00\x0e\x00\x00\x01\x1e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTR\x00\x00\x00\x01\x00\x00\x0e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x15\x00\x00\x15com.apple.quarantine\x00\x00\x00\x00\x00\x00\x00\x00\x00%com.apple.metadata:kMDItemWhereFroms\x000081;699ccfcc;Chrome;bplist00\x01\x02_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download_\x10?https://openneuro.org/datasets/ds005691/versions/1.0.0/download\x08\x0bM\x00\x00\x00\x00\x00\x00\x01\x01\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

OK [32/32] brainvision | shape (7, 9456550) | sfreq 2500.0

Dataset caricati: 8


In [2]:
# ============================================================
# NeuroCore Feature Extractor v3
# Mass extraction cell
# ============================================================

import numpy as np

WINDOW_SEC = 2


def extract_neurocore(file_struct):

    data = file_struct["data"]
    sfreq = file_struct["sfreq"]

    window = int(WINDOW_SEC * sfreq)

    if window < 10:
        window = 10

    n_windows = data.shape[1] // window

    if n_windows < 2:
        return None

    data = data[:, :n_windows * window]

    windows = data.reshape(
        data.shape[0],
        n_windows,
        window
    ).transpose(1,0,2)


    # ===== core NeuroCore primitives =====

    mean = np.mean(windows, axis=2)

    var = np.var(windows, axis=2)

    diff = np.diff(mean, axis=0)

    transitions = np.sum(np.abs(diff), axis=1)

    acceleration = np.diff(transitions)

    resilience = 1 / (1 + np.var(transitions))

    lambda_signature = np.mean(np.abs(acceleration))

    dynamic_pressure = np.mean(transitions)

    transition_ratio = np.mean(transitions) / (
        np.std(transitions) + 1e-8
    )

    return {

        "file": file_struct["file"],

        "shape": data.shape,

        "sfreq": sfreq,

        "windows": n_windows,

        "channels": data.shape[0],

        "resilience": float(resilience),

        "lambda": float(lambda_signature),

        "dynamic_pressure": float(dynamic_pressure),

        "transition_ratio": float(transition_ratio),

        "var_mean": float(np.mean(var)),

        "var_std": float(np.std(var))
    }



# === ESTRAZIONE COMPLETA ===

results = []

for f in dataset:

    r = extract_neurocore(f)

    if r is not None:
        results.append(r)


print("\nFILES PROCESSATI:", len(results))


# stampa esempio
for r in results[:5]:

    print("\n---")
    print("File:", r["file"])
    print("Lambda:", r["lambda"])
    print("Resilience:", r["resilience"])
    print("Dynamic Pressure:", r["dynamic_pressure"])


FILES PROCESSATI: 8

---
File: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-01/ieeg/sub-01_task-countingDeviants_ieeg.vhdr
Lambda: 6.234493048395962e-05
Resilience: 1.0
Dynamic Pressure: 0.00011154181265737861

---
File: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-02/ieeg/sub-02_task-countingDeviants_ieeg.vhdr
Lambda: 4.149908272665925e-05
Resilience: 1.0
Dynamic Pressure: 7.329313666559756e-05

---
File: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-03/ieeg/sub-03_task-countingDeviants_ieeg.vhdr
Lambda: 5.217776197241619e-05
Resilience: 1.0
Dynamic Pressure: 0.0001106563868233934

---
File: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/SpinalExpect_Invasive_005691/sub-04/ieeg/sub-04_task-countingDeviants_ieeg.vhdr
Lambda: 3.3195348805747926e-05
Resilience: 1.0
Dynamic Pressure: 6.653950549662113e-05

---
File: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/Spi

In [3]:
# ============================================================
# NeuroCore Universal Regime Analysis Cell v2 (Fixed AUC)
# Works with ANY NeuroCore dataset present or future
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4

# ============================================================
# USER CONFIGURATION
# ============================================================

OUTPUT_DRIVE = "/Volumes/KINGSTON"   # Modifica se serve
REPORT_NAME = "NeuroCore_Report"
WINDOW_SEC = 2

# ============================================================
# SAFETY CHECK
# ============================================================

if "dataset" not in globals():
    raise Exception("dataset variable not found. Run NeuroCore loader first.")

# ============================================================
# FEATURE EXTRACTION
# ============================================================

results = []

for file_struct in dataset:
    try:
        data = file_struct["data"]
        sfreq = file_struct["sfreq"]

        window = int(WINDOW_SEC * sfreq)
        if window < 10:
            window = 10

        n_windows = data.shape[1] // window
        if n_windows < 3:
            continue

        data_cut = data[:, :n_windows*window]

        windows = data_cut.reshape(data_cut.shape[0], n_windows, window).transpose(1,0,2)

        mean = np.mean(windows, axis=2)
        diff = np.diff(mean, axis=0)
        transitions = np.sum(np.abs(diff), axis=1)

        dynamic_pressure = float(np.mean(transitions))
        resilience = float(1 / (1 + np.var(transitions)))
        lambda_sig = float(np.mean(np.abs(np.diff(transitions))))

        results.append({
            "file": file_struct["file"],
            "dynamic_pressure": dynamic_pressure,
            "resilience": resilience,
            "lambda": lambda_sig,
            "channels": data.shape[0],
            "samples": data.shape[1],
            "sfreq": sfreq
        })

    except Exception as e:
        print("Failed:", file_struct["file"], str(e))

if len(results) == 0:
    raise Exception("No valid files processed.")

# ============================================================
# SORT TRANSITIONS
# ============================================================

df = pd.DataFrame(results)
df = df.sort_values("dynamic_pressure")
transitions_sorted = df["dynamic_pressure"].values

# ============================================================
# AUC COMPUTATION (Universal)
# ============================================================

try:
    auc = np.trapezoid(transitions_sorted)
except AttributeError:
    auc = np.trapz(transitions_sorted)

# ============================================================
# CREATE OUTPUT FOLDER
# ============================================================

output_folder = os.path.join(OUTPUT_DRIVE, "NeuroCore")
os.makedirs(output_folder, exist_ok=True)

# ============================================================
# SAVE CSV
# ============================================================

csv_path = os.path.join(output_folder, REPORT_NAME + ".csv")
df["AUC_total"] = auc
df.to_csv(csv_path, index=False)

# ============================================================
# PLOT REGIME TRANSITIONS
# ============================================================

plt.figure()
plt.plot(transitions_sorted)
plt.title("NeuroCore Regime Transitions")
plt.xlabel("Ordered Files")
plt.ylabel("Dynamic Pressure")
plt.tight_layout()

plot_path = os.path.join(output_folder, REPORT_NAME + "_plot.png")
plt.savefig(plot_path)
plt.close()

# ============================================================
# PDF REPORT
# ============================================================

pdf_path = os.path.join(output_folder, REPORT_NAME + ".pdf")
styles = getSampleStyleSheet()
doc = SimpleDocTemplate(pdf_path, pagesize=A4)
elements = []

# TITLE
elements.append(Paragraph("NeuroCore Regime Analysis Report", styles["Heading1"]))
elements.append(Spacer(1,12))

# SUMMARY
elements.append(Paragraph(f"Files analyzed: {len(df)}", styles["Normal"]))
elements.append(Paragraph(f"AUC: {auc:.10f}", styles["Normal"]))
elements.append(Spacer(1,12))

# GRAPH
elements.append(Image(plot_path, width=6*inch, height=4*inch))
elements.append(Spacer(1,12))

# TABLE
table_data = [["File", "Dynamic Pressure", "Resilience", "Lambda"]]
for _, row in df.iterrows():
    table_data.append([
        os.path.basename(row["file"]),
        f"{row['dynamic_pressure']:.6f}",
        f"{row['resilience']:.6f}",
        f"{row['lambda']:.6f}"
    ])

table = Table(table_data[:25])
table.setStyle(TableStyle([
    ('GRID',(0,0),(-1,-1),0.5,colors.grey),
    ('BACKGROUND',(0,0),(-1,0),colors.lightgrey)
]))
elements.append(table)

doc.build(elements)

# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n===== NeuroCore Report Generated =====")
print("Files analyzed:", len(df))
print("AUC:", auc)
print("CSV:", csv_path)
print("PDF:", pdf_path)
print("Plot:", plot_path)


===== NeuroCore Report Generated =====
Files analyzed: 8
AUC: 0.0005877916337340139
CSV: /Volumes/KINGSTON/NeuroCore/NeuroCore_Report.csv
PDF: /Volumes/KINGSTON/NeuroCore/NeuroCore_Report.pdf
Plot: /Volumes/KINGSTON/NeuroCore/NeuroCore_Report_plot.png
